# 🚀 WORKER COLAB: SINH VIDEO LTX-VIDEO (BẢN BỀN v4)
Chạy lần lượt từ trên xuống: **Ô 1 → 2 → 3 → 4 → 5 → 6**. Pipeline hai pha (T5 → LTX-Video) trên Tesla T4:

1. **Pha 1:** mã hóa prompt bằng T5, lưu cache, rồi giải phóng bộ nhớ.
2. **Pha 2:** sinh từng clip 704x384 @ 24fps, **kiểm tra video sau khi ghi**, tự thử lại / hạ số khung hình nếu lỗi.

**Lần đầu nên chạy thử 1 shot** (ô 5, điền `ONLY = "shot_01"`), thấy OK rồi mới chạy cả hàng đợi.
Nếu Colab ngắt kết nối: mở lại, chạy lại từ ô 1. Shot đã xong sẽ được bỏ qua (nếu đã gắn Google Drive thì clip còn nguyên trên Drive).

In [ ]:
#@title ✅ 1. Kiểm tra GPU
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    raise RuntimeError("❌ Không có GPU. Vào Runtime → Change runtime type → chọn T4 GPU, rồi chạy lại từ ô 1.")
print("GPU:", out.stdout.strip())

In [ ]:
#@title 📦 2. Cài thư viện (KHÔNG cài lại torch, dùng bản có sẵn của Colab)
!pip install -q -U "diffusers>=0.32.0" "transformers>=4.46.0" "accelerate>=1.0.0" sentencepiece protobuf "imageio[ffmpeg]" imageio-ffmpeg psutil

# OpenCV thường có sẵn trên Colab; chỉ cài nếu thiếu
try:
    import cv2
except ImportError:
    !pip install -q opencv-python-headless

# Kiểm tra những gì worker thực sự cần
import importlib
for name in ["torch", "diffusers", "transformers", "accelerate", "sentencepiece", "imageio", "imageio_ffmpeg", "cv2", "psutil"]:
    try:
        m = importlib.import_module(name)
        print(f"✅ {name} {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"❌ {name}: {e}")
from diffusers import LTXImageToVideoPipeline   # nếu dòng này lỗi: Runtime → Restart session rồi chạy lại ô 2
print("✅ LTXImageToVideoPipeline sẵn sàng")

In [ ]:
#@title 💾 3. (Khuyến nghị) Gắn Google Drive để không mất clip khi Colab ngắt
MIRROR_DIR = ""
try:
    from google.colab import drive
    drive.mount("/content/drive")
    MIRROR_DIR = "/content/drive/MyDrive/Create-Video-output"
    print("✅ Clip xong sẽ được chép sang:", MIRROR_DIR)
except Exception as e:
    print("ℹ️ Bỏ qua Google Drive (clip chỉ nằm trên máy ảo Colab):", e)

In [ ]:
#@title 📥 4. Tải mã nguồn mới nhất từ GitHub
import os
WORKSPACE_DIR = "/content/Create-Video"
if not os.path.exists(os.path.join(WORKSPACE_DIR, ".git")):
    !git clone https://github.com/SieuHuynh-glitch/Create-Video.git {WORKSPACE_DIR}
else:
    # reset --hard để không bao giờ bị kẹt vì thay đổi cục bộ (khác với git pull)
    !git -C {WORKSPACE_DIR} fetch origin && git -C {WORKSPACE_DIR} reset --hard origin/master
%cd {WORKSPACE_DIR}

# Tự kiểm tra: bản worker mới đã có trên GitHub chưa?
src = open("scripts/colab_worker.py", encoding="utf-8").read()
if "WORKER_VERSION" not in src:
    raise RuntimeError("❌ GitHub vẫn đang là worker CŨ (chưa push bản sửa). Hãy commit và push scripts/colab_worker.py rồi chạy lại ô 4.")
!python scripts/colab_worker.py --version

In [ ]:
#@title 🤖 5. Chạy Worker
SLUG = "natural-cycle"  #@param {type:"string"}
ONLY = ""               #@param {type:"string"}
# ONLY: để trống = chạy tất cả shot. Chạy thử: "shot_01". Nhiều shot: "shot_01 shot_05"

cmd = f"python scripts/colab_worker.py --slug {SLUG}"
if ONLY.strip():
    cmd += f" --only {ONLY.strip()}"
if MIRROR_DIR:
    cmd += f" --mirror-dir {MIRROR_DIR}"
print("▶️", cmd)
!{cmd}

In [ ]:
#@title 📦 6. Kiểm tra clip, đóng gói ZIP và tải về
import zipfile, json, cv2
from pathlib import Path
from google.colab import files

done_dir = Path(f"/content/Create-Video/jobs/done/{SLUG}")
report = Path(f"/content/Create-Video/jobs/report-{SLUG}.json")
if report.exists():
    summary = json.loads(report.read_text(encoding="utf-8"))["summary"]
    print("📋 Kết quả lần chạy gần nhất:", summary)

good = []
for f in sorted(done_dir.glob("shot_*.mp4")):
    cap = cv2.VideoCapture(str(f))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    status = "✅" if n > 0 else "❌ HỎNG"
    print(f"{status} {f.name}: {n} khung, {f.stat().st_size / 1024 / 1024:.2f} MB")
    if n > 0:
        good.append(f)

failed = sorted(Path(f"/content/Create-Video/jobs/failed/{SLUG}").glob("*.json"))
for f in failed:
    print(f"⚠️ {f.stem} còn lỗi: {json.loads(f.read_text(encoding='utf-8')).get('error')}")

if not good:
    raise RuntimeError("Chưa có clip hợp lệ nào để đóng gói. Xem log ô 5.")

zip_path = Path(f"/content/{SLUG}-clips.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in good:
        z.write(f, arcname=f.name)
print(f"\n🎉 Đã nén {len(good)} clip → {zip_path.name} ({zip_path.stat().st_size / 1024 / 1024:.1f} MB)")
files.download(str(zip_path))

### 🛟 Khi gặp sự cố
| Triệu chứng | Cách xử lý |
|---|---|
| `Không có GPU` | Runtime → Change runtime type → **T4 GPU**, chạy lại từ ô 1 |
| Ô 2 báo lỗi import | Runtime → **Restart session**, rồi chạy lại từ ô 1 |
| Ô 4 báo "worker CŨ" | Bản sửa chưa được push lên GitHub: commit + push `scripts/colab_worker.py` |
| Hết VRAM / bộ nhớ | Runtime → Restart session. Worker tự thử lại với ít khung hình hơn và ghi rõ trong báo cáo |
| Một shot lỗi | Các shot khác vẫn chạy tiếp. Lý do nằm trong `jobs/failed/<slug>/shot_xx.json`. Chạy lại ô 5 để thử lại shot đó |
| Muốn sinh lại một shot | Điền `ONLY = "shot_03"` và thêm `--force` vào lệnh ở ô 5 |
| Colab ngắt giữa chừng | Mở lại, chạy lại từ ô 1. Shot đã xong được bỏ qua |